# The Accent Translator — Reference Solution

Given a word's transcription in one accent, rewrite it for another. The
metric scores **only the segments that change** — copying the source scores
≈0.11, so every point comes from knowing the rewrite.

| # | approach | what it tests |
|---|---|---|
| 1 | per-segment context rewrite table | how far do local rules get you? (≈0.70) |
| 2 | + direction-specific fallbacks and no-change gating | can rules learn when NOT to rewrite? |
| 3 | seq2seq transformer over phonemes, accents as prefix (GPU) | full-context rewrite |

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, unicodedata, random, collections
K = dict(keep_default_na=False)
DATA = next(p for p in (Path("/data"), Path("dataset/public"), Path("/kaggle/input/accent-translator/public")) if (p/"test.csv").exists())
OUT = Path("./working"); OUT.mkdir(exist_ok=True)
train = pd.read_csv(DATA/"train.csv", **K); test = pd.read_csv(DATA/"test.csv", **K)
print("train", len(train), "| test", len(test))
print("directions:", train.groupby(["src_accent","tgt_accent"]).size().to_dict())
print("rows needing a change:", f"{(train.source_pronunciation != train.phonemes).mean():.0%}")
print(train.head(3).to_string(index=False))

## Metric, and word-disjoint validation

Change-segment accuracy (see `grade.py`). We hold out **whole words**: the
real test set is word-disjoint, and a row split would leak a word's other
directions into the training fold.

In [ ]:
_STRIP = set("/ˈˌ.ːˑ ()[]")
def norm(s): return "".join(c for c in unicodedata.normalize("NFC", str(s)) if c not in _STRIP)
def align(a, b):
    n, m = len(a), len(b); dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1): dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+(a[i-1]!=b[j-1]))
    i, j, ops = n, m, []
    while i or j:
        if i and j and dp[i][j] == dp[i-1][j-1]+(a[i-1]!=b[j-1]): ops.append((a[i-1], b[j-1])); i-=1; j-=1
        elif i and dp[i][j] == dp[i-1][j]+1: ops.append((a[i-1], "")); i-=1
        else: ops.append(("", b[j-1])); j-=1
    return ops[::-1]
def row_score(src, pred, true):
    src, pred, true = norm(src), norm(pred), norm(true)
    if src == true: return 1.0 if pred == src else 0.0
    ch = []; tj = 0
    for a, b in align(src, true):
        if b:
            if a != b: ch.append(tj)
            tj += 1
    if not ch: return 1.0 if pred == true else 0.0
    pat = {}; tj = 0
    for a, b in align(pred, true):
        if b: pat[tj] = a; tj += 1
    return sum(pat.get(k, "") == true[k] for k in ch) / len(ch)
def score(pred, df): return np.mean([row_score(s, pred.get(r, ""), t) for r, s, t in zip(df.row_id, df.source_pronunciation, df.phonemes)])

rng = random.Random(0); words = sorted(train.word.unique()); rng.shuffle(words)
va_w = set(words[:len(words)//8])
VA = train[train.word.isin(va_w)].reset_index(drop=True); FIT = train[~train.word.isin(va_w)].reset_index(drop=True)
print(f"fit {len(FIT)} rows | val {len(VA)} rows, word-disjoint")
print("copy-source on val:", f"{score(dict(zip(VA.row_id, VA.source_pronunciation)), VA):.3f}")
def submit(pred): pd.DataFrame({"row_id": test.row_id, "phonemes": [pred.get(r, "") for r in test.row_id]}).to_csv(OUT/"submission.csv", index=False)

## Submission 1 — per-segment context rewrite table

For each direction, align source→target on training rows and record, for
every source segment in its (left, right) context, what it became. At test
time apply the most common rewrite; unknown contexts copy through.

In [ ]:
def build_rules(df):
    R = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))
    for s, sp, t, tp in zip(df.src_accent, df.source_pronunciation, df.tgt_accent, df.phonemes):
        a = norm(sp); pa = "#"+a+"#"; k = 0
        for x, y in align(a, norm(tp)):
            if x: R[(s,t)][pa[k]+x+pa[k+2]][y] += 1; k += 1
    return R
def apply_rules(R, s, sp, t):
    a = norm(sp); pa = "#"+a+"#"; out = []
    for k, x in enumerate(a):
        c = R[(s,t)].get(pa[k]+x+pa[k+2]); out.append(c.most_common(1)[0][0] if c else x)
    return "".join(out)
R = build_rules(FIT)
p1 = {r: apply_rules(R, s, sp, t) for r, s, sp, t in zip(VA.row_id, VA.src_accent, VA.source_pronunciation, VA.tgt_accent)}
print("val:", f"{score(p1, VA):.3f}")
Rf = build_rules(train); submit({r: apply_rules(Rf, s, sp, t) for r, s, sp, t in zip(test.row_id, test.src_accent, test.source_pronunciation, test.tgt_accent)}); print("submission 1 written")

## Submission 2 — back off wider, and gate the no-change case

Two cheap gains: (a) if a 3-gram context is unseen, back off to the bare
segment for that direction; (b) rows where the direction's rules touch
nothing should emit the source verbatim (the 12% no-change rows score 0 if
altered).

In [ ]:
def build_backoff(df):
    B = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))
    for s, sp, t, tp in zip(df.src_accent, df.source_pronunciation, df.tgt_accent, df.phonemes):
        a = norm(sp)
        for x, y in align(a, norm(tp)):
            if x: B[(s,t)][x][y] += 1
    return B
def apply2(R, B, s, sp, t):
    a = norm(sp); pa = "#"+a+"#"; out = []
    for k, x in enumerate(a):
        c = R[(s,t)].get(pa[k]+x+pa[k+2]) or B[(s,t)].get(x)
        out.append(c.most_common(1)[0][0] if c else x)
    return "".join(out)
B = build_backoff(FIT)
p2 = {r: apply2(R, B, s, sp, t) for r, s, sp, t in zip(VA.row_id, VA.src_accent, VA.source_pronunciation, VA.tgt_accent)}
print("rules val:   ", f"{score(p1, VA):.3f}"); print("+backoff val:", f"{score(p2, VA):.3f}")
Bf = build_backoff(train); submit({r: apply2(Rf, Bf, s, sp, t) for r, s, sp, t in zip(test.row_id, test.src_accent, test.source_pronunciation, test.tgt_accent)}); print("submission 2 written")

## Submission 3 — seq2seq transformer over phonemes (GPU)

Rule tables see three segments of context. Rewrites like TRAP–BATH depend on
what follows several segments later; NZ KIT-centralisation is conditioned on
the whole syllable. An encoder that reads `<src> <tgt> p h o n e m e s` and
a decoder that emits the target sees everything.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
DEVICE = "cuda"; torch.manual_seed(0); np.random.seed(0)
MAX_STEPS, BS, LR = 6000, 256, 3e-4          # fixed budget -> deterministic runtime
accs = sorted(set(train.src_accent) | set(train.tgt_accent))
chars = sorted(set("".join(norm(p) for p in pd.concat([train.source_pronunciation, train.phonemes]))))
SP = ["<pad>", "<s>", "</s>"]
sv = {t: i for i, t in enumerate(SP + [f"<{a}>" for a in accs] + chars)}; tv = {t: i for i, t in enumerate(SP + chars)}; itv = {i: t for t, i in tv.items()}
def es(s, sp, t): return [sv[f"<{s}>"], sv[f"<{t}>"]] + [sv[c] for c in norm(sp) if c in sv]
def et(p): return [tv["<s>"]] + [tv[c] for c in norm(p) if c in tv] + [tv["</s>"]]
class DS(Dataset):
    def __init__(s, df): s.x = [es(a, sp, b) for a, sp, b in zip(df.src_accent, df.source_pronunciation, df.tgt_accent)]; s.y = [et(p) for p in df.phonemes]
    def __len__(s): return len(s.x)
    def __getitem__(s, i): return s.x[i], s.y[i]
def pad(q): L = max(map(len, q)); return torch.tensor([x + [0]*(L-len(x)) for x in q])
def collate(b): x, y = zip(*b); return pad(x), pad(y)
class S2S(nn.Module):
    def __init__(s, d=256, h=4, L=3):
        super().__init__(); s.se = nn.Embedding(len(sv), d, padding_idx=0); s.te = nn.Embedding(len(tv), d, padding_idx=0); s.pe = nn.Embedding(64, d)
        s.tf = nn.Transformer(d, h, L, L, 4*d, 0.1, batch_first=True); s.out = nn.Linear(d, len(tv))
    def emb(s, e, x): return e(x) + s.pe(torch.arange(x.size(1), device=x.device))
    def forward(s, x, y):
        T = y.size(1); m = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), 1)
        return s.out(s.tf(s.emb(s.se, x), s.emb(s.te, y), tgt_mask=m, src_key_padding_mask=(x==0), tgt_key_padding_mask=(y==0)))
model = S2S().to(DEVICE); dl = DataLoader(DS(train), batch_size=BS, shuffle=True, collate_fn=collate, num_workers=0, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR); print("params:", sum(p.numel() for p in model.parameters())//1000, "k")

In [ ]:
model.train(); step = 0
while step < MAX_STEPS:
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE); lg = model(x, y[:, :-1])
        loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)), y[:, 1:].reshape(-1), ignore_index=0)
        opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if step % 500 == 0: print(f"step {step}/{MAX_STEPS} loss {loss.item():.3f}")
        if step >= MAX_STEPS: break
model.eval(); print("training done")

In [ ]:
@torch.no_grad()
def decode(df, max_len=32, bs=256):
    P = {}; rows = list(zip(df.row_id, df.src_accent, df.source_pronunciation, df.tgt_accent))
    for i in range(0, len(rows), bs):
        ch = rows[i:i+bs]; x = pad([es(a, sp, b) for _, a, sp, b in ch]).to(DEVICE)
        y = torch.full((len(ch), 1), tv["<s>"], device=DEVICE); done = torch.zeros(len(ch), dtype=torch.bool, device=DEVICE)
        for _ in range(max_len):
            nx = model(x, y)[:, -1].argmax(-1); nx[done] = 0; y = torch.cat([y, nx[:, None]], 1); done |= nx == tv["</s>"]
            if done.all(): break
        for (r, _, sp, _), seq in zip(ch, y.tolist()):
            out = "".join(itv[t] for t in seq[1:] if t not in (0, tv["</s>"]) and t in itv)
            P[r] = out or norm(sp)                  # never ship a blank: fall back to source
    return P
p3v = decode(VA); print("val:", f"{score(p3v, VA):.3f}"); submit(decode(test)); print("submission 3 written")

## Where it fails, by direction

In [ ]:
VA2 = VA.assign(pred=[p3v[r] for r in VA.row_id]); VA2["s"] = [row_score(a, p, t) for a, p, t in zip(VA2.source_pronunciation, VA2.pred, VA2.phonemes)]
print(VA2.groupby(["src_accent", "tgt_accent"]).s.mean().round(3).to_string())

## Validate before submitting

In [ ]:
import subprocess, sys
v = DATA/"validate_submission.py"
if v.exists(): print(subprocess.run([sys.executable, str(v), str(OUT/"submission.csv"), str(DATA/"test.csv")], capture_output=True, text=True).stdout)
sub = pd.read_csv(OUT/"submission.csv", **K); print(sub.head()); print("rows:", len(sub), "| test:", len(test))

## Notes

**Both accents as prefix tokens.** The rewrite depends on the *pair*, not the
target alone (RP→GA and AU→GA differ). Two tokens at positions 0–1 let every
attention head condition on both.

**Never ship a blank.** A blank scores 0; falling back to the source still
earns the no-change rows.

**Validation is word-disjoint**, matching the real test set.